## 📰 RSS Feed Collection -- Colombian News Outlets

### By:
MGO

### Date:
2026-03-22

### Description:

Collect political news articles from major Colombian news outlets via RSS feeds.
This is the first and most reliable data source -- no API keys required, just HTTP requests
to public RSS/Atom feeds. We collect from El Tiempo, El Espectador, Semana, La Silla Vacia,
Caracol Radio, Blu Radio, Razon Publica, and others.

**Data source priority:** P0 (Day 1)

**Output:** `data/01_raw/rss/rss_articles_YYYYMMDD.json`

## 📚 Import libraries

In [ ]:
from __future__ import annotations

import json
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd

# Add src to path for importing project utilities
sys.path.insert(0, str(Path("../../src").resolve()))

from collectors.rss import collect_feeds_from_config
from utils.config import load_config
from utils.normalize import normalize_post

## ⚙️ Configuration

Load RSS feed URLs from the YAML config file. This keeps the feed list
separate from the code -- a non-engineer can add or remove feeds by editing
`conf/data_collection/rss_feeds.yml`.

In [ ]:
# Load feed configuration
config = load_config("../../conf/data_collection/rss_feeds.yml")
feeds = config["feeds"]
settings = config["settings"]

print(f"Configured {len(feeds)} RSS feeds:")
for feed in feeds:
    print(f"  - {feed['name']}: {feed['url']}")

print(f"\nDelay between requests: {settings['request_delay_seconds']}s")
print(f"Max articles per feed: {settings['max_articles_per_feed']}")

## 💾 Data Collection

Use `feedparser` to fetch and parse each RSS feed. The `collect_feeds_from_config`
helper handles the iteration and respects the configured delay between requests.

In [ ]:
# Collect articles from all configured feeds
print("Starting RSS collection...")
articles = collect_feeds_from_config(
    feeds_config=feeds,
    delay=settings["request_delay_seconds"],
    settings=settings,
)
active_feeds = [f for f in feeds if f.get("enabled", True)]
print(
    f"Collected {len(articles)} articles from {len(set(a.get('source', '') for a in articles))} feeds"
)

In [ ]:
# Convert to DataFrame for inspection
df_raw = pd.DataFrame(articles)
print(f"Shape: {df_raw.shape}")
print(f"\nColumns: {df_raw.columns.tolist()}")
df_raw.head()

## 👷 Data Normalization

Normalize each article to the common post schema used across all data sources.
This ensures that downstream analysis notebooks can work with a consistent format
regardless of whether the data came from RSS, Reddit, or other sources.

In [ ]:
# Normalize to common schema
normalized = [normalize_post(article, source="rss") for article in articles]
df_normalized = pd.DataFrame(normalized)
print(f"Normalized {len(df_normalized)} articles")
print(f"\nCommon schema columns: {df_normalized.columns.tolist()}")
df_normalized[["id", "text", "source", "platform", "timestamp"]].head()

## 💾 Save to Raw Layer

Save the raw collected data to `data/01_raw/rss/` as a dated JSON file.
The raw layer is immutable -- we never modify files here, only append new ones.

In [ ]:
# Save raw articles
today = datetime.now().strftime("%Y%m%d")
output_dir = Path("../../data/01_raw/rss")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / f"rss_articles_{today}.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(articles, f, ensure_ascii=False, indent=2, default=str)

print(f"Saved {len(articles)} articles to {output_path}")

## 📊 Collection Summary

In [ ]:
# Summary statistics
print("=" * 60)
print("RSS COLLECTION SUMMARY")
print("=" * 60)
print(f"Total articles collected: {len(articles)}")
print(f"Unique sources: {df_raw['source'].nunique()}")
print("\nArticles per source:")
print(df_raw["source"].value_counts().to_string())

if "published" in df_raw.columns:
    print(f"\nDate range: {df_raw['published'].min()} to {df_raw['published'].max()}")

print("\nSample article titles:")
for title in df_raw["title"].head(5):
    print(f"  - {title[:80]}")

## 💡 Notes & Next Steps

**Quality observations:**
- Some feeds may return articles without a `published` date -- these need special handling
- HTML content in `summary` fields should be cleaned before NLP processing
- Categories/tags from RSS can be useful for pre-filtering political vs. non-political content

**Next steps:**
1. Run normalization to `data/02_intermediate/` with common schema
2. Filter for political content using keyword matching
3. Feed into sentiment analysis pipeline (notebook 3-analysis/01_mgo_sentiment_analysis)

**Scheduling:**
- This collection should run every 1 hour to capture breaking news
- Use `make collect_rss` or the Makefile target for automated runs

## 📖 References

- feedparser documentation: https://feedparser.readthedocs.io/
- El Tiempo RSS: https://www.eltiempo.com/rss/
- CIVICUS Monitor Colombia: https://monitor.civicus.org/country/colombia/